
# Price Elasticity Analysis

**Objective:** Analysis and validation of final price elasticities.


## 📚 1. Libraries and Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import os
import plotly.express as px
from scipy.stats import pearsonr

# Visual configuration
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
sns.set_theme(style="whitegrid", palette="pastel")

import warnings
warnings.filterwarnings('ignore')

# Pandas formatting
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.precision', 2)


## 📥 2. Data Import and Preparation

> **Technical Note:** The process below uses a **pagination strategy (Seek Method)** to process large volumes of promotional campaign history without exceeding driver memory, performing complex joins between sales tables and promotional calendars.

In [ ]:
from databricks import sql
import pyspark.sql.functions as F
from pyspark.sql import DataFrame
import functools

def process_large_scale_data():
    # ======================================================================================
    # STEP 1: Connection Setup and Base View
    # ======================================================================================

    # Credentials retrieved via environment variables for security
    DB_HOST = os.getenv("DB_HOST_PROD")
    DB_PATH = os.getenv("DB_WAREHOUSE_PATH")
    DB_TOKEN = os.getenv("DB_ACCESS_TOKEN")

    print("Connecting to Data Warehouse...")
    connection = sql.connect(
        server_hostname=DB_HOST,
        http_path=DB_PATH,
        access_token=DB_TOKEN)
    cursor = connection.cursor()

    # Prepare sales transaction base for the join
    print("Generating Sales TempView...")
    sales_query = """
    SELECT
        t1.*,
        t2.ORDER_DATE,
        t2.STORE_ID
    FROM
        sales.transaction_flat_table AS t1
    LEFT JOIN
        (SELECT
            ORDER_ID,
            STORE_ID,
            MAX(ORDER_DATE) AS ORDER_DATE
        FROM
            sales.refined_product_analysis
        GROUP BY
            ORDER_ID,
            STORE_ID) AS t2 ON t1.ORDER_ID = t2.ORDER_ID
    """
    df_sales = spark.sql(sales_query)
    
    # Add unique ID for granularity control
    df_sales_id = df_sales.withColumn("unique_row_id", F.monotonically_increasing_id())
    df_sales_id.createOrReplaceTempView("sales_view")
    print("View 'sales_view' ready.")

    # ======================================================================================
    # STEP 2: Iterative Processing (Seek Method) for Campaigns
    # ======================================================================================

    try:
        chunk_size = 20000  
        results_list = []
        last_campaign_id = 0 # Cursor for pagination

        while True:
            # Fetch incremental batch of active campaigns
            query_pagination = f"""
            SELECT campaign_pk, campaign_id, start_date, end_date, target_store_ids 
            FROM marketing.campaign_history_v
            WHERE start_date >= '2024-01-01' AND campaign_pk > {last_campaign_id}
            ORDER BY campaign_pk ASC
            LIMIT {chunk_size}
            """
            
            print(f"Processing batch after ID {last_campaign_id}...")
            cursor.execute(query_pagination)
            campaign_chunk = cursor.fetchall()

            if not campaign_chunk:
                print("Campaign ingestion finished.")
                break

            last_campaign_id = campaign_chunk[-1][0]

            # Explode store list (comma-separated string) into rows
            exploded_rows = []
            for row in campaign_chunk:
                camp_pk, camp_id, start, end, stores_str = row
                if stores_str:
                    for store in stores_str.split(','):
                        if store:
                            exploded_rows.append((camp_pk, camp_id, start, end, store))
            
            if exploded_rows:
                # Create temporary DataFrame for optimized join
                df_chunk = spark.createDataFrame(
                    exploded_rows,
                    schema=["campaign_pk", "campaign_id", "start_date", "end_date", "STORE_ID_TARGET"])
                df_chunk.createOrReplaceTempView("campaign_chunk_view")

                # Join by date range and store
                match_query = """
                SELECT
                    s.unique_row_id,
                    c.campaign_pk,
                    c.campaign_id
                FROM sales_view s
                JOIN campaign_chunk_view c 
                    ON s.STORE_ID = c.STORE_ID_TARGET
                    AND s.ORDER_DATE BETWEEN CAST(c.start_date AS DATE) AND CAST(c.end_date AS DATE)
                """
                df_matches = spark.sql(match_query)
                
                # Cache partial result
                results_list.append(df_matches.cache())
                print(f"Batch finished at {last_campaign_id}: {df_matches.count()} matches.")

        # ======================================================================================
        # STEP 3: Consolidation and Persistence
        # ======================================================================================
        print("\nConsolidating results...")
        if results_list:
            df_all_matches = functools.reduce(DataFrame.unionAll, results_list)

            # Group matches by sale (one sale can have multiple campaigns)
            df_aggregated = df_all_matches.groupBy("unique_row_id").agg(
                F.collect_list("campaign_pk").alias("campaign_pk_list"),
                F.collect_list("campaign_id").alias("campaign_id_list"))

            df_final_joined = df_sales_id.join(df_aggregated, "unique_row_id", "left")
            
            # Feature engineering: boolean flag for active campaign
            df_final_elasticity = df_final_joined \
                .withColumn("is_promotion_active", F.size(F.coalesce(F.col("campaign_pk_list"), F.array())) > 0) \
                .drop("unique_row_id")

            target_table = "analytics.price_elasticity_analysis_final"
            df_final_elasticity.write.mode("overwrite").saveAsTable(target_table)
            print(f"Table saved: {target_table}")

        else:
            print("No matches found.")

    finally:
        cursor.close()
        connection.close()
        print("Connection closed.")

# Commented out for on-demand execution only
# process_large_scale_data()

In [ ]:
# Load processed table for analysis
df_elasticity = spark.table("analytics.price_elasticity_analysis_final")
df_elasticity_pd = df_elasticity.toPandas()

In [ ]:
df_elasticity_pd.info()

In [ ]:
# Retrieve model metadata via MLflow for comparison
run_id = "0e2f073cc60441d3a82ad06d0a2eb95d" # Generic ID

local_path = mlflow.artifacts.download_artifacts(
    run_id=run_id,
    artifact_path="diagnostics/model_summary_extended.csv")

df_diagnostics = pd.read_csv(local_path)
df_diagnostics.info()


## 📊 3. Exploratory Analysis and Correlations

In [ ]:
display(df_elasticity)

In [ ]:
# Descriptive statistics of target variable
df_elasticity_pd['price_elasticity'].describe()

In [ ]:
# Convert types for visualization
viz_df = df_elasticity.select("is_promotion_active", "price_elasticity").toPandas()
viz_df['is_promotion_active'] = viz_df['is_promotion_active'].astype('category')

plt.figure(figsize=(10, 6))

sns.boxplot(
    x="price_elasticity",
    y="is_promotion_active",
    data=viz_df)

plt.xlabel("Price Elasticity")
plt.ylabel("Active Campaign?")
plt.title("Elasticity Distribution by Promotional Status")
plt.tight_layout()
plt.show()

In [ ]:
# Proportion analysis of positive/negative elasticity via PySpark
result_stats = (
    df_elasticity.withColumn("elasticity_sign", 
                             F.when(F.col("price_elasticity") > 0, "positive").otherwise("negative"))
      .groupBy("is_promotion_active", "elasticity_sign")
      .agg(F.count("*").alias("count")))

total_group = (
    df_elasticity.groupBy("is_promotion_active")
      .agg(F.count("*").alias("total")))

# Calculate representativeness
proportion_df = (
    result_stats.join(total_group, on="is_promotion_active")
          .withColumn("proportion", F.col("count") / F.col("total"))
          .select("is_promotion_active", "elasticity_sign", "count", "total", "proportion"))

display(proportion_df)

In [ ]:
print('-'*40)
print('Global Correlation: Quantity vs Log(Price)')
print('-'*40)

# Global Pearson correlation test
corr_coef, p_val = pearsonr(df_elasticity_pd['quantity_sold'], df_elasticity_pd['log_price'])

print(f"Pearson Coefficient: {corr_coef:.4f}")
print(f"P-value: {p_val:.4f}")

if p_val < 0.05:
    print("Statistically significant result (p < 0.05).")
else:
    print("Not statistically significant (p >= 0.05).")

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(df_elasticity_pd['price_elasticity'], kde=True)
plt.axvline(x=0, color='red', linestyle='--', linewidth=2)

plt.xlabel('Price Elasticity')
plt.ylabel('Frequency')
plt.title('Price Elasticity Distribution')
plt.show()

In [ ]:
# Check class balance
df_elasticity_pd['is_promotion_active'].value_counts(normalize=True)

In [ ]:
# Categorical variables for granular analysis
analysis_vars = ['brand_name',
    'product_category',
    'sales_channel',
    'sub_channel',
    'state_region',
    'region_macro',
    'business_unit',
    'sub_business_unit',
    'sales_team',
    'reference_month']

# Generate boxplots for each dimension
for var in analysis_vars:
    plt.figure(figsize=(10, 6))
    sns.boxplot(
        data=df_elasticity_pd,
        x='price_elasticity',
        y=var,
        orient='h')
    plt.title(f'Elasticity by {var}', fontsize=14, pad=12)
    plt.xlabel('Elasticity', fontsize=12)
    plt.ylabel(var, fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
print("=" * 60)
print(f"Detailed Correlation Analysis (Qty vs Log Price)")
print("=" * 60)
print('\n')

def analyze_correlations(full_df, group_col):
    """
    Calculates Pearson and p-value for subgroups and returns metadata.
    """
    results = []

    print(f"Grouping: '{group_col}'")
    print("=" * 60)

    for category, group_df in full_df.groupby(group_col):
        # Validation to avoid errors in very small groups or zero variance
        if len(group_df) < 2:
            continue
            
        corr, p_val = pearsonr(group_df['quantity_sold'], group_df['log_price'])
        
        results.append({
            'Variable': group_col,
            'Category': category,
            'Pearson_Corr': corr,
            'P_Value': p_val
        })

        print(f"\nCategory: {category}")
        print(f"  - Pearson....: {corr:.4f}")
        print(f"  - P-value....: {p_val:.4f}")
        print(f"  - N..........: {len(group_df)}")

        if p_val < 0.05:
            print("  - Significant: Yes")
        else:
            print("  - Significant: No")

    print('\n')
    return results

# Run correlation loop
all_correlations = []
for v in analysis_vars:
    stats_data = analyze_correlations(df_elasticity_pd, v)
    all_correlations.extend(stats_data)

In [ ]:
df_corr_results = pd.DataFrame(all_correlations)
df_corr_results.head()

In [ ]:
# Calculate mean elasticity by category to cross-reference with correlation
mean_elasticity_list = []

for var in analysis_vars:
    mean_by_cat = df_elasticity_pd.groupby(var)['price_elasticity'].mean()
    df_mean = mean_by_cat.reset_index()
    
    df_mean.rename(columns={
        var: 'Category',
        'price_elasticity': 'Mean_Elasticity'
    }, inplace=True)
    
    df_mean['Variable'] = var
    mean_elasticity_list.append(df_mean)

df_means_consolidated = pd.concat(mean_elasticity_list, ignore_index=True)

In [ ]:
# Enrich correlation dataframe
df_corr_results = df_corr_results.merge(
    df_means_consolidated[['Variable', 'Category', 'Mean_Elasticity']],
    on=['Variable', 'Category'],
    how='left')
display(df_corr_results)

In [ ]:
mlflow.autolog(disable=True)

# Interactive Scatterplot: Mean Elasticity vs Correlation
fig = px.scatter(
    data_frame=df_corr_results,
    y='Mean_Elasticity',
    x='Pearson_Corr',
    color='Variable',
    hover_data=['Category', 'Mean_Elasticity'],
    trendline='ols',
    title='Relationship: Mean Elasticity vs Correlation Strength (Qty x Price)',
    labels={
        'Mean_Elasticity': 'Mean Elasticity',
        'Pearson_Corr': 'Pearson Correlation (Qty vs Log Price)',
        'Variable': 'Analyzed Variable'
    },
    template='plotly_white')

fig.add_hline(y=0, line_dash="dot", line_color="grey")
fig.show()

In [ ]:
mlflow.autolog(disable=True)

# Stratified sampling by Sub Business Unit (1%) for lightweight visualization
df_sampled = (
    df_elasticity_pd
    .groupby('sub_business_unit', group_keys=False)
    .sample(frac=0.01, random_state=42)
    .reset_index(drop=True)
    .copy())

fig = px.scatter(
    data_frame=df_sampled,
    y='quantity_sold',
    x='log_price',
    color='sub_business_unit',
    hover_data=analysis_vars + ['price_elasticity'],
    trendline='ols',
    title='Demand Curve: Qty vs Log Price by Sub-BU',
    labels={
        'quantity_sold': 'Volume Sold',
        'log_price': 'Log Unit Price',
        'sub_business_unit': 'Sub Business Unit'
    },
    opacity=0.6,
    template='plotly_white',
    marginal_y='histogram')

fig.add_hline(y=0, line_dash="dot", line_color="grey")
fig.show()

In [ ]:
mlflow.autolog(disable=True)

# Sampling by Brand
df_sampled_brand = (
    df_elasticity_pd
    .groupby('brand_name', group_keys=False)
    .sample(frac=0.01, random_state=42)
    .reset_index(drop=True)
    .copy())

fig = px.scatter(
    data_frame=df_sampled_brand,
    y='quantity_sold',
    x='log_price',
    color='brand_name',
    hover_data=analysis_vars + ['price_elasticity'],
    trendline='ols',
    title='Demand Curve by Brand',
    labels={
        'quantity_sold': 'Volume Sold',
        'log_price': 'Log Unit Price',
        'brand_name': 'Brand'
    },
    opacity=0.6,
    template='plotly_white',
    marginal_y='histogram')

fig.add_hline(y=0, line_dash="dot", line_color="grey")
fig.show()

In [ ]:
# Final visualization of model diagnostics
display(df_diagnostics)